## 🧪 Preprocesamiento previo al análisis (descriptivo ~ predictivo/agrupativo)

In [2]:
# =================================================================
# 🔱 Iniciar la SparkSession 🔱
# =================================================================
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd

# Configuración de Hadoop para leer .parquet
os.environ["HADOOP_HOME"] = r"C:\\hadoop"
os.environ["hadoop.home.dir"] = r"C:\\hadoop"

# Crear la SparkSession
spark = SparkSession.builder \
    .appName("previous_preprocessing") \
    .getOrCreate()

print("✅ SparkSession iniciada correctamente.")

✅ SparkSession iniciada correctamente.


# ⚜️ Admitidos ⚜️

In [3]:
# Carga de Admitidos (.parquet)
df_adm = spark.read.parquet("C:\\Users\\sebas\\OneDrive\\Desktop\\Proyecto_Final_BigDataV\\data\\lake\\admitidos")

df_adm.printSchema()
df_adm.show(5)
print("Cantidad de admitidos:", df_adm.count())

root
 |-- CÓDIGO DE LA INSTITUCIÓN: string (nullable = true)
 |-- IES PADRE: string (nullable = true)
 |-- INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES): string (nullable = true)
 |-- TIPO IES: string (nullable = true)
 |-- ID SECTOR IES: string (nullable = true)
 |-- SECTOR IES: string (nullable = true)
 |-- ID CARÁCTER IES: string (nullable = true)
 |-- CARÁCTER IES: string (nullable = true)
 |-- IES ACREDITADA: string (nullable = true)
 |-- PRINCIPAL O SECCIONAL: string (nullable = true)
 |-- CÓDIGO DEL DEPARTAMENTO (IES): string (nullable = true)
 |-- DEPARTAMENTO DE DOMICILIO DE LA IES: string (nullable = true)
 |-- CÓDIGO DEL MUNICIPIO IES: string (nullable = true)
 |-- MUNICIPIO DE DOMICILIO DE LA IES: string (nullable = true)
 |-- CÓDIGO SNIES DEL PROGRAMA: string (nullable = true)
 |-- PROGRAMA ACADÉMICO: string (nullable = true)
 |-- PROGRAMA ACREDITADO: string (nullable = true)
 |-- ID NIVEL ACADÉMICO: string (nullable = true)
 |-- NIVEL ACADÉMICO: string (nullable = true)
 |-- ID

In [4]:
# Primero observar la cantidad de nulos de cada columna
null_counts = df_adm.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df_adm.columns])
null_counts.show()

# Luego, la cantidad de registros duplicados
total_rows = df_adm.count()
distinct_rows = df_adm.distinct().count()
duplicates = total_rows - distinct_rows

print(f"Filas totales:    {total_rows}")
print(f"Filas únicas:     {distinct_rows}")
print(f"Filas duplicadas: {duplicates}")

+------------------------+---------+---------------------------------------+--------+-------------+----------+---------------+------------+--------------+---------------------+-----------------------------+-----------------------------------+------------------------+--------------------------------+-------------------------+------------------+-------------------+------------------+---------------+---------------------+------------------+------------+---------+-------+--------------------+---------+------------------------------------+--------------------+----------------------+------------------------+--------------------------+-----------------------+-------------------------+----------------------------------+-----------------------------------+-------------------------------+--------------------------------+-------+----+--------+---------+---+
|CÓDIGO DE LA INSTITUCIÓN|IES PADRE|INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)|TIPO IES|ID SECTOR IES|SECTOR IES|ID CARÁCTER IES|CARÁCTER IES|IE

In [5]:
## Eliminar columnas con nulos mayores al 35% 
total_rows = df_adm.count()
threshold = 0.35  # Porcentaje máximo permitido de nulos

# Calcular porcentaje de nulos por columna
null_percentages = [
    (c, df_adm.filter(F.col(c).isNull()).count() / total_rows)
    for c in df_adm.columns
]

# Seleccionar columnas que cumplen con el umbral
cols_to_keep = [c for c, perc in null_percentages if perc <= threshold]

# Crear nuevo DataFrame sin las columnas con demasiados nulos
df_adm_clean = df_adm.select(cols_to_keep)

In [6]:
## Imputar con 0 nulos en admitidos para no alterar
df_adm_clean = df_adm_clean.fillna({"ADMITIDOS": 0})

# Eliminar filas donde falte año y semestre
df_adm_clean = df_adm_clean.filter(
    F.col("AÑO").isNotNull() &
    F.col("SEMESTRE").isNotNull()
)

# Sector IES nulos son desconocidos
df_adm_clean = df_adm_clean.fillna({
    "SECTOR IES": "DESCONOCIDO"
})

In [7]:
# Verificación de nulos en el DF limpio
null_counts = df_adm_clean.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df_adm_clean.columns])
null_counts.show()

print("✅ Preprocesamiento de admitidos completado.")

+------------------------+---------+---------------------------------------+--------+-------------+----------+---------------+------------+---------------------+-----------------------------+-----------------------------------+------------------------+--------------------------------+-------------------------+------------------+------------------+---------------+---------------------+------------------+------------+---------+-------+--------------------+---------+------------------------------------+----------------------------------+-----------------------------------+-------------------------------+--------------------------------+-------+----+--------+---------+---+
|CÓDIGO DE LA INSTITUCIÓN|IES PADRE|INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)|TIPO IES|ID SECTOR IES|SECTOR IES|ID CARÁCTER IES|CARÁCTER IES|PRINCIPAL O SECCIONAL|CÓDIGO DEL DEPARTAMENTO (IES)|DEPARTAMENTO DE DOMICILIO DE LA IES|CÓDIGO DEL MUNICIPIO IES|MUNICIPIO DE DOMICILIO DE LA IES|CÓDIGO SNIES DEL PROGRAMA|PROGRAMA AC

# ⚜️ Docentes ⚜️

In [8]:
# Carga de Docentes (.parquet)
df_doc = spark.read.parquet("C:\\Users\\sebas\\OneDrive\\Desktop\\Proyecto_Final_BigDataV\\data\\lake\\docentes")

df_doc.printSchema()
df_doc.show(5)
print("Cantidad de docentes:", df_doc.count())

root
 |-- CÓDIGO DE LA INSTITUCIÓN: string (nullable = true)
 |-- IES PADRE: string (nullable = true)
 |-- INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES): string (nullable = true)
 |-- TIPO IES: string (nullable = true)
 |-- ID SECTOR IES: string (nullable = true)
 |-- SECTOR IES: string (nullable = true)
 |-- ID CARÁCTER IES: string (nullable = true)
 |-- CARÁCTER IES: string (nullable = true)
 |-- IES ACREDITADA: string (nullable = true)
 |-- PRINCIPAL O SECCIONAL: string (nullable = true)
 |-- CÓDIGO DEL DEPARTAMENTO (IES): string (nullable = true)
 |-- DEPARTAMENTO DE DOMICILIO DE LA IES: string (nullable = true)
 |-- CÓDIGO DEL MUNICIPIO IES: string (nullable = true)
 |-- MUNICIPIO DE DOMICILIO DE LA IES: string (nullable = true)
 |-- ID SEXO: string (nullable = true)
 |-- SEXO: string (nullable = true)
 |-- TIPO DE DOCUMENTO: string (nullable = true)
 |-- ID MÁXIMO NIVEL DE FORMACIÓN DEL DOCENTE: string (nullable = true)
 |-- MÁXIMO NIVEL DE FORMACIÓN DEL DOCENTE: string (nullable = tru

In [9]:
# Primero observar la cantidad de nulos de cada columna
null_counts = df_doc.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df_doc.columns])
null_counts.show()

# Luego, la cantidad de registros duplicados
total_rows = df_doc.count()
distinct_rows = df_doc.distinct().count()
duplicates = total_rows - distinct_rows

print(f"Filas totales:    {total_rows}")
print(f"Filas únicas:     {distinct_rows}")
print(f"Filas duplicadas: {duplicates}")

+------------------------+---------+---------------------------------------+--------+-------------+----------+---------------+------------+--------------+---------------------+-----------------------------+-----------------------------------+------------------------+--------------------------------+-------+----+-----------------+----------------------------------------+-------------------------------------+-----------------------+--------------------------------+-------------------+----------------------------+--------+---------------+----------------------------------+----------------------------------+----------------------------------+----------+---+
|CÓDIGO DE LA INSTITUCIÓN|IES PADRE|INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)|TIPO IES|ID SECTOR IES|SECTOR IES|ID CARÁCTER IES|CARÁCTER IES|IES ACREDITADA|PRINCIPAL O SECCIONAL|CÓDIGO DEL DEPARTAMENTO (IES)|DEPARTAMENTO DE DOMICILIO DE LA IES|CÓDIGO DEL MUNICIPIO IES|MUNICIPIO DE DOMICILIO DE LA IES|ID SEXO|SEXO|TIPO DE DOCUMENTO|ID MÁXI

In [10]:
## Eliminar columnas con nulos mayores al 35% 
total_rows = df_doc.count()
threshold = 0.35   # Porcentaje máximo permitido de nulos

# Calcular porcentaje de nulos por columna
null_percentages = [
    (c, df_doc.filter(F.col(c).isNull()).count() / total_rows)
    for c in df_doc.columns
]

# Seleccionar columnas que cumplen con el umbral
cols_to_keep = [c for c, perc in null_percentages if perc <= threshold]

# Crear nuevo DataFrame sin las columnas con demasiados nulos
df_doc_clean = df_doc.select(cols_to_keep)

In [11]:
# Eliminar filas donde falte año y semestre
df_doc_clean = df_doc_clean.filter(
    F.col("AÑO").isNotNull() &
    F.col("SEMESTRE").isNotNull()
)

# Sector IES nulos son desconocidos
df_doc_clean = df_doc_clean.fillna({
    "SECTOR IES": "DESCONOCIDO"
})

In [12]:
# Verificación de nulos en el DF limpio
null_counts = df_doc_clean.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df_doc_clean.columns])
null_counts.show()

print("✅ Preprocesamiento de docentes completado.")

+------------------------+---------+---------------------------------------+--------+-------------+----------+---------------+------------+---------------------+-----------------------------+-----------------------------------+------------------------+--------------------------------+-------+----+----------------------------------------+-------------------------------------+-----------------------+--------------------------------+-------------------+----------------------------+--------+---------------+---+
|CÓDIGO DE LA INSTITUCIÓN|IES PADRE|INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)|TIPO IES|ID SECTOR IES|SECTOR IES|ID CARÁCTER IES|CARÁCTER IES|PRINCIPAL O SECCIONAL|CÓDIGO DEL DEPARTAMENTO (IES)|DEPARTAMENTO DE DOMICILIO DE LA IES|CÓDIGO DEL MUNICIPIO IES|MUNICIPIO DE DOMICILIO DE LA IES|ID SEXO|SEXO|ID MÁXIMO NIVEL DE FORMACIÓN DEL DOCENTE|MÁXIMO NIVEL DE FORMACIÓN DEL DOCENTE|ID TIEMPO DE DEDICACIÓN|TIEMPO DE DEDICACIÓN DEL DOCENTE|ID TIPO DE CONTRATO|TIPO DE CONTRATO DEL DOCENTE|SEME

# ⚜️ Graduados ⚜️

In [13]:
# Carga de Graduados (.parquet)
df_grad = spark.read.parquet("C:\\Users\\sebas\\OneDrive\\Desktop\\Proyecto_Final_BigDataV\\data\\lake\\graduados")

df_grad.printSchema()
df_grad.show(5)
print("Cantidad de graduados: ", df_grad.count())

root
 |-- CÓDIGO DE LA INSTITUCIÓN: string (nullable = true)
 |-- IES PADRE: string (nullable = true)
 |-- INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES): string (nullable = true)
 |-- TIPO IES: string (nullable = true)
 |-- ID SECTOR IES: string (nullable = true)
 |-- SECTOR IES: string (nullable = true)
 |-- ID CARÁCTER IES: string (nullable = true)
 |-- CARÁCTER IES: string (nullable = true)
 |-- IES ACREDITADA: string (nullable = true)
 |-- PRINCIPAL O SECCIONAL: string (nullable = true)
 |-- CÓDIGO DEL DEPARTAMENTO (IES): string (nullable = true)
 |-- DEPARTAMENTO DE DOMICILIO DE LA IES: string (nullable = true)
 |-- CÓDIGO DEL MUNICIPIO IES: string (nullable = true)
 |-- MUNICIPIO DE DOMICILIO DE LA IES: string (nullable = true)
 |-- CÓDIGO SNIES DEL PROGRAMA: string (nullable = true)
 |-- PROGRAMA ACADÉMICO: string (nullable = true)
 |-- PROGRAMA ACREDITADO: string (nullable = true)
 |-- ID NIVEL ACADÉMICO: string (nullable = true)
 |-- NIVEL ACADÉMICO: string (nullable = true)
 |-- ID

In [14]:
# Primero observar la cantidad de nulos de cada columna
null_counts = df_grad.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df_grad.columns])
null_counts.show()

# Luego, la cantidad de registros duplicados
total_rows = df_grad.count()
distinct_rows = df_grad.distinct().count()
duplicates = total_rows - distinct_rows

print(f"Filas totales:    {total_rows}")
print(f"Filas únicas:     {distinct_rows}")
print(f"Filas duplicadas: {duplicates}")

+------------------------+---------+---------------------------------------+--------+-------------+----------+---------------+------------+--------------+---------------------+-----------------------------+-----------------------------------+------------------------+--------------------------------+-------------------------+------------------+-------------------+------------------+---------------+---------------------+------------------+------------+---------+-------+--------------------+---------+------------------------------------+--------------------+----------------------+------------------------+--------------------------+-----------------------+-------------------------+----------------------------------+-----------------------------------+-------------------------------+--------------------------------+-------+----+--------+---------+--------------------------+---+
|CÓDIGO DE LA INSTITUCIÓN|IES PADRE|INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)|TIPO IES|ID SECTOR IES|SECTOR IES|ID C

In [15]:
## Eliminar columnas con nulos mayores al 35% 
total_rows = df_grad.count()
threshold = 0.35   # Porcentaje máximo permitido de nulos

# Calcular porcentaje de nulos por columna
null_percentages = [
    (c, df_grad.filter(F.col(c).isNull()).count() / total_rows)
    for c in df_grad.columns
]

# Seleccionar columnas que cumplen con el umbral
cols_to_keep = [c for c, perc in null_percentages if perc <= threshold]

# Crear nuevo DataFrame sin las columnas con demasiados nulos
df_grad_clean = df_grad.select(cols_to_keep)

In [16]:
# Eliminar filas donde falte año y semestre
df_grad_clean = df_grad_clean.filter(
    F.col("AÑO").isNotNull() &
    F.col("SEMESTRE").isNotNull()
)

# Sector IES nulos son desconocidos
df_grad_clean = df_grad_clean.fillna({
    "SECTOR IES": "DESCONOCIDO"
})

In [17]:
# Verificación de nulos en el DF limpio
null_counts = df_grad_clean.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df_grad_clean.columns])
null_counts.show()

print("✅ Preprocesamiento de graduados completado.")

+------------------------+---------+---------------------------------------+--------+-------------+----------+---------------+------------+---------------------+-----------------------------+-----------------------------------+------------------------+--------------------------------+-------------------------+------------------+------------------+---------------+---------------------+------------------+------------+---------+-------+--------------------+---------+------------------------------------+----------------------------------+-----------------------------------+-------------------------------+--------------------------------+-------+----+--------+---------+---+
|CÓDIGO DE LA INSTITUCIÓN|IES PADRE|INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)|TIPO IES|ID SECTOR IES|SECTOR IES|ID CARÁCTER IES|CARÁCTER IES|PRINCIPAL O SECCIONAL|CÓDIGO DEL DEPARTAMENTO (IES)|DEPARTAMENTO DE DOMICILIO DE LA IES|CÓDIGO DEL MUNICIPIO IES|MUNICIPIO DE DOMICILIO DE LA IES|CÓDIGO SNIES DEL PROGRAMA|PROGRAMA AC

# ⚜️ Inscritos ⚜️

In [18]:
# Carga de Inscritos (.parquet)
df_insc = spark.read.parquet("C:\\Users\\sebas\\OneDrive\\Desktop\\Proyecto_Final_BigDataV\\data\\lake\\inscritos")

df_insc.printSchema()
df_insc.show(5)
print("Cantidad de inscritos: ", df_insc.count())

root
 |-- CÓDIGO DE LA INSTITUCIÓN: string (nullable = true)
 |-- IES PADRE: string (nullable = true)
 |-- INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES): string (nullable = true)
 |-- TIPO IES: string (nullable = true)
 |-- ID SECTOR IES: string (nullable = true)
 |-- SECTOR IES: string (nullable = true)
 |-- ID CARÁCTER IES: string (nullable = true)
 |-- CARÁCTER IES: string (nullable = true)
 |-- IES ACREDITADA: string (nullable = true)
 |-- PRINCIPAL O SECCIONAL: string (nullable = true)
 |-- CÓDIGO DEL DEPARTAMENTO (IES): string (nullable = true)
 |-- DEPARTAMENTO DE DOMICILIO DE LA IES: string (nullable = true)
 |-- CÓDIGO DEL MUNICIPIO IES: string (nullable = true)
 |-- MUNICIPIO DE DOMICILIO DE LA IES: string (nullable = true)
 |-- CÓDIGO SNIES DEL PROGRAMA: string (nullable = true)
 |-- PROGRAMA ACADÉMICO: string (nullable = true)
 |-- PROGRAMA ACREDITADO: string (nullable = true)
 |-- ID NIVEL ACADÉMICO: string (nullable = true)
 |-- NIVEL ACADÉMICO: string (nullable = true)
 |-- ID

In [19]:
# Primero observar la cantidad de nulos de cada columna
null_counts = df_insc.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df_insc.columns])
null_counts.show()

# Luego, la cantidad de registros duplicados
total_rows = df_insc.count()
distinct_rows = df_insc.distinct().count()
duplicates = total_rows - distinct_rows

print(f"Filas totales:    {total_rows}")
print(f"Filas únicas:     {distinct_rows}")
print(f"Filas duplicadas: {duplicates}")

+------------------------+---------+---------------------------------------+--------+-------------+----------+---------------+------------+--------------+---------------------+-----------------------------+-----------------------------------+------------------------+--------------------------------+-------------------------+------------------+-------------------+------------------+---------------+---------------------+------------------+------------+---------+-------+--------------------+---------+------------------------------------+--------------------+----------------------+------------------------+--------------------------+-----------------------+-------------------------+----------------------------------+-----------------------------------+-------------------------------+--------------------------------+-------+----+--------+---------+---+
|CÓDIGO DE LA INSTITUCIÓN|IES PADRE|INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)|TIPO IES|ID SECTOR IES|SECTOR IES|ID CARÁCTER IES|CARÁCTER IES|IE

In [20]:
## Eliminar columnas con nulos mayores al 35% 
total_rows = df_insc.count()
threshold = 0.35   # Porcentaje máximo permitido de nulos

# Calcular porcentaje de nulos por columna
null_percentages = [
    (c, df_insc.filter(F.col(c).isNull()).count() / total_rows)
    for c in df_insc.columns
]

# Seleccionar columnas que cumplen con el umbral
cols_to_keep = [c for c, perc in null_percentages if perc <= threshold]

# Crear nuevo DataFrame sin las columnas con demasiados nulos
df_insc_clean = df_insc.select(cols_to_keep)

In [21]:
# Eliminar filas donde falte año y semestre
df_insc_clean = df_insc_clean.filter(
    F.col("AÑO").isNotNull() &
    F.col("SEMESTRE").isNotNull()
)

# Sector IES nulos son desconocidos
df_insc_clean = df_insc_clean.fillna({
    "SECTOR IES": "DESCONOCIDO"
})

In [22]:
# Verificación de nulos en el DF limpio
null_counts = df_insc_clean.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df_insc_clean.columns])
null_counts.show()

print("✅ Preprocesamiento de inscritos completado.")

+------------------------+---------+---------------------------------------+--------+-------------+----------+---------------+------------+---------------------+-----------------------------+-----------------------------------+------------------------+--------------------------------+-------------------------+------------------+------------------+---------------+---------------------+------------------+------------+---------+-------+--------------------+---------+------------------------------------+----------------------------------+-----------------------------------+-------------------------------+--------------------------------+-------+----+--------+---------+---+
|CÓDIGO DE LA INSTITUCIÓN|IES PADRE|INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)|TIPO IES|ID SECTOR IES|SECTOR IES|ID CARÁCTER IES|CARÁCTER IES|PRINCIPAL O SECCIONAL|CÓDIGO DEL DEPARTAMENTO (IES)|DEPARTAMENTO DE DOMICILIO DE LA IES|CÓDIGO DEL MUNICIPIO IES|MUNICIPIO DE DOMICILIO DE LA IES|CÓDIGO SNIES DEL PROGRAMA|PROGRAMA AC

### ⚜️ Matriculados ⚜️

In [23]:
# Carga de Matriculados (.parquet)
df_matr = spark.read.parquet("C:\\Users\\sebas\\OneDrive\\Desktop\\Proyecto_Final_BigDataV\\data\\lake\\matriculados")

df_matr.printSchema()
df_matr.show(5)
print("Cantidad de matriculados: ", df_matr.count())

root
 |-- CÓDIGO DE LA INSTITUCIÓN: string (nullable = true)
 |-- IES PADRE: string (nullable = true)
 |-- INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES): string (nullable = true)
 |-- TIPO IES: string (nullable = true)
 |-- ID SECTOR IES: string (nullable = true)
 |-- SECTOR IES: string (nullable = true)
 |-- ID CARÁCTER IES: string (nullable = true)
 |-- CARÁCTER IES: string (nullable = true)
 |-- IES ACREDITADA: string (nullable = true)
 |-- PRINCIPAL O SECCIONAL: string (nullable = true)
 |-- CÓDIGO DEL DEPARTAMENTO (IES): string (nullable = true)
 |-- DEPARTAMENTO DE DOMICILIO DE LA IES: string (nullable = true)
 |-- CÓDIGO DEL MUNICIPIO IES: string (nullable = true)
 |-- MUNICIPIO DE DOMICILIO DE LA IES: string (nullable = true)
 |-- CÓDIGO SNIES DEL PROGRAMA: string (nullable = true)
 |-- PROGRAMA ACADÉMICO: string (nullable = true)
 |-- PROGRAMA ACREDITADO: string (nullable = true)
 |-- ID NIVEL ACADÉMICO: string (nullable = true)
 |-- NIVEL ACADÉMICO: string (nullable = true)
 |-- ID

In [24]:
# Primero observar la cantidad de nulos de cada columna
null_counts = df_matr.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df_matr.columns])
null_counts.show()

# Luego, la cantidad de registros duplicados
total_rows = df_matr.count()
distinct_rows = df_matr.distinct().count()
duplicates = total_rows - distinct_rows

print(f"Filas totales:    {total_rows}")
print(f"Filas únicas:     {distinct_rows}")
print(f"Filas duplicadas: {duplicates}")

+------------------------+---------+---------------------------------------+--------+-------------+----------+---------------+------------+--------------+---------------------+-----------------------------+-----------------------------------+------------------------+--------------------------------+-------------------------+------------------+-------------------+------------------+---------------+---------------------+------------------+------------+---------+-------+--------------------+---------+------------------------------------+--------------------+----------------------+------------------------+--------------------------+-----------------------+-------------------------+----------------------------------+-----------------------------------+-------------------------------+--------------------------------+-------+----+--------+------------+---+
|CÓDIGO DE LA INSTITUCIÓN|IES PADRE|INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)|TIPO IES|ID SECTOR IES|SECTOR IES|ID CARÁCTER IES|CARÁCTER IES

In [25]:
## Eliminar columnas con nulos mayores al 35% 
total_rows = df_matr.count()
threshold = 0.35   # Porcentaje máximo permitido de nulos

# Calcular porcentaje de nulos por columna
null_percentages = [
    (c, df_matr.filter(F.col(c).isNull()).count() / total_rows)
    for c in df_matr.columns
]

# Seleccionar columnas que cumplen con el umbral
cols_to_keep = [c for c, perc in null_percentages if perc <= threshold]

# Crear nuevo DataFrame sin las columnas con demasiados nulos
df_matr_clean = df_matr.select(cols_to_keep)

In [26]:
# Eliminar filas donde falte año y semestre
df_matr_clean = df_matr_clean.filter(
    F.col("AÑO").isNotNull() &
    F.col("SEMESTRE").isNotNull()
)

# Sector IES nulos son desconocidos
df_matr_clean = df_matr_clean.fillna({
    "SECTOR IES": "DESCONOCIDO"
})

In [27]:
# Verificación de nulos en el DF limpio
null_counts = df_matr_clean.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df_matr_clean.columns])
null_counts.show()

print("✅ Preprocesamiento de matriculados completado.")

+------------------------+---------+---------------------------------------+--------+-------------+----------+---------------+------------+---------------------+-----------------------------+-----------------------------------+------------------------+--------------------------------+-------------------------+------------------+------------------+---------------+---------------------+------------------+------------+---------+-------+--------------------+---------+------------------------------------+----------------------------------+-----------------------------------+-------------------------------+--------------------------------+-------+----+--------+------------+---+
|CÓDIGO DE LA INSTITUCIÓN|IES PADRE|INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)|TIPO IES|ID SECTOR IES|SECTOR IES|ID CARÁCTER IES|CARÁCTER IES|PRINCIPAL O SECCIONAL|CÓDIGO DEL DEPARTAMENTO (IES)|DEPARTAMENTO DE DOMICILIO DE LA IES|CÓDIGO DEL MUNICIPIO IES|MUNICIPIO DE DOMICILIO DE LA IES|CÓDIGO SNIES DEL PROGRAMA|PROGRAMA

### ⚜️ Matriculados Primer Curso ⚜️

In [28]:
# Carga de Matriculados primer curso (.parquet)
df_matr_curso = spark.read.parquet("C:\\Users\\sebas\\OneDrive\\Desktop\\Proyecto_Final_BigDataV\\data\\lake\\matriculados_primer_anho")

df_matr_curso.printSchema()
df_matr_curso.show(5)
print("Cantidad de matriculados en primer curso: ", df_matr_curso.count())

root
 |-- CÓDIGO DE LA INSTITUCIÓN: string (nullable = true)
 |-- IES PADRE: string (nullable = true)
 |-- INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES): string (nullable = true)
 |-- TIPO IES: string (nullable = true)
 |-- ID SECTOR IES: string (nullable = true)
 |-- SECTOR IES: string (nullable = true)
 |-- ID CARÁCTER IES: string (nullable = true)
 |-- CARÁCTER IES: string (nullable = true)
 |-- IES ACREDITADA: string (nullable = true)
 |-- PRINCIPAL O SECCIONAL: string (nullable = true)
 |-- CÓDIGO DEL DEPARTAMENTO (IES): string (nullable = true)
 |-- DEPARTAMENTO DE DOMICILIO DE LA IES: string (nullable = true)
 |-- CÓDIGO DEL MUNICIPIO IES: string (nullable = true)
 |-- MUNICIPIO DE DOMICILIO DE LA IES: string (nullable = true)
 |-- CÓDIGO SNIES DEL PROGRAMA: string (nullable = true)
 |-- PROGRAMA ACADÉMICO: string (nullable = true)
 |-- PROGRAMA ACREDITADO: string (nullable = true)
 |-- ID NIVEL ACADÉMICO: string (nullable = true)
 |-- NIVEL ACADÉMICO: string (nullable = true)
 |-- ID

In [29]:
# Primero observar la cantidad de nulos de cada columna
null_counts = df_matr_curso.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df_matr_curso.columns])
null_counts.show()

# Luego, la cantidad de registros duplicados
total_rows = df_matr_curso.count()
distinct_rows = df_matr_curso.distinct().count()
duplicates = total_rows - distinct_rows

print(f"Filas totales:    {total_rows}")
print(f"Filas únicas:     {distinct_rows}")
print(f"Filas duplicadas: {duplicates}")

+------------------------+---------+---------------------------------------+--------+-------------+----------+---------------+------------+--------------+---------------------+-----------------------------+-----------------------------------+------------------------+--------------------------------+-------------------------+------------------+-------------------+------------------+---------------+---------------------+------------------+------------+---------+-------+--------------------+---------+------------------------------------+--------------------+----------------------+------------------------+--------------------------+-----------------------+-------------------------+----------------------------------+-----------------------------------+-------------------------------+--------------------------------+-------+-----+--------+-------------------------+---+
|CÓDIGO DE LA INSTITUCIÓN|IES PADRE|INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)|TIPO IES|ID SECTOR IES|SECTOR IES|ID CARÁCTER IE

In [30]:
## Eliminar columnas con nulos mayores al 35% 
total_rows = df_matr_curso.count()
threshold = 0.35   # Porcentaje máximo permitido de nulos

# Calcular porcentaje de nulos por columna
null_percentages = [
    (c, df_matr_curso.filter(F.col(c).isNull()).count() / total_rows)
    for c in df_matr_curso.columns
]

# Seleccionar columnas que cumplen con el umbral
cols_to_keep = [c for c, perc in null_percentages if perc <= threshold]

# Crear nuevo DataFrame sin las columnas con demasiados nulos
df_matr_curso_clean = df_matr_curso.select(cols_to_keep)

In [31]:
# Eliminar filas donde falte año y semestre
df_matr_curso_clean = df_matr_curso_clean.filter(
    F.col("AÑO").isNotNull() &
    F.col("SEMESTRE").isNotNull()
)

# Sector IES nulos son desconocidos
df_matr_curso_clean = df_matr_curso_clean.fillna({
    "SECTOR IES": "DESCONOCIDO",
    "SEXO": "DESCONOCIDO"
})

In [32]:
# Verificación de nulos en el DF limpio
null_counts = df_matr_curso_clean.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df_matr_curso_clean.columns])
null_counts.show()

print("✅ Preprocesamiento de matriculados en su primer curso completado.")

+------------------------+---------+---------------------------------------+--------+-------------+----------+---------------+------------+---------------------+-----------------------------+-----------------------------------+------------------------+--------------------------------+-------------------------+------------------+------------------+---------------+---------------------+------------------+------------+---------+-------+--------------------+---------+------------------------------------+----------------------------------+-----------------------------------+-------------------------------+--------------------------------+-------+----+--------+-------------------------+---+
|CÓDIGO DE LA INSTITUCIÓN|IES PADRE|INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)|TIPO IES|ID SECTOR IES|SECTOR IES|ID CARÁCTER IES|CARÁCTER IES|PRINCIPAL O SECCIONAL|CÓDIGO DEL DEPARTAMENTO (IES)|DEPARTAMENTO DE DOMICILIO DE LA IES|CÓDIGO DEL MUNICIPIO IES|MUNICIPIO DE DOMICILIO DE LA IES|CÓDIGO SNIES DEL PROG

# 🎲 Preprocesamiento previo Finalizado exitosamente 🎲

## 🔮 Guardar datos en data\processed en formato .csv

In [33]:
from pathlib import Path

output_root = Path(r"C:\\Users\\sebas\\OneDrive\\Desktop\\Proyecto_Final_BigDataV\\data\\processed")
output_root.mkdir(parents=True, exist_ok=True)

dataframes_limpios = {
    "admitidos": df_adm_clean,
    "docentes": df_doc_clean,
    "graduados": df_grad_clean,
    "inscritos": df_insc_clean,
    "matriculados": df_matr_clean,
    "matriculados_primer_curso": df_matr_curso_clean,
}

for nombre, sdf in dataframes_limpios.items():
    output_file = output_root / f"{nombre}.csv"

    pdf = sdf.toPandas()
    pdf.to_csv(output_file, index=False, encoding="utf-8-sig")

    print(f" ✅ Guardado: {output_file}")


 ✅ Guardado: C:\Users\sebas\OneDrive\Desktop\Proyecto_Final_BigDataV\data\processed\admitidos.csv
 ✅ Guardado: C:\Users\sebas\OneDrive\Desktop\Proyecto_Final_BigDataV\data\processed\docentes.csv
 ✅ Guardado: C:\Users\sebas\OneDrive\Desktop\Proyecto_Final_BigDataV\data\processed\graduados.csv
 ✅ Guardado: C:\Users\sebas\OneDrive\Desktop\Proyecto_Final_BigDataV\data\processed\inscritos.csv
 ✅ Guardado: C:\Users\sebas\OneDrive\Desktop\Proyecto_Final_BigDataV\data\processed\matriculados.csv
 ✅ Guardado: C:\Users\sebas\OneDrive\Desktop\Proyecto_Final_BigDataV\data\processed\matriculados_primer_curso.csv
